<h2 style="text-align:center;">Optimal Execution<br></h2>
<div style="text-align:center;">Ariel Kalingking</div>
<div style="text-align:center;">akalingking@gmail.com</div>
<p style="text-align:center;">Appendix Python Code</p>

In [121]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import cm
#import matplotlib.ticker as mticker
import logging
#import io
#import base64

imagepath = "../paper/figures/"
matplotlib.use('TkAgg')
logging.getLogger().setLevel(logging.ERROR)
figsize = (4,3)
fontbig = 8
fontmed = 7
fontsmall = 6
# Unset when being built from makefile
plt_inline_enable = False
if plt_inline_enable:
    %matplotlib inline

In [117]:
### Model parameters ###
eta = 0.1                  # Risk aversion for trading (eta > 0)
gamma = 0.01               # Risk aversion for inventory (gamma > 0)
sigma = 0.05               # Volatility of the price process (sigma > 0)
T = 1.0                    # Time horizon (e.g., 1 day)

# Time Discretization
N_t = 200                  # Number of time steps (backward)
t_min, t_max = 0, T        # Boundary condition for time
dt = T / N_t               # Time step size
print("dt:", dt)

# Inventory Discretization
N_q = 100                  # Number of inventory grid points
q_min, q_max = -500, 500    # Boundary condition for inventory size
dq = (q_max - q_min) / N_q  # Inventory step size

dt: 0.005


In [118]:
#
# Numerical Approximation using derivatives 
# for finding optimal trading rate, v*(q,t).
#
# Create discrete grids
inventory_grid = np.linspace(q_min, q_max, N_q)
time_grid = np.linspace(t_min, t_max, N_t)

# V_grid[N_t, :] is already initialized to zeros.
V_grid = np.zeros((N_t, N_q))
V_star = np.zeros((N_t, N_q)) # Store v* at each (t, q) point

# Set Terminal Condition, V(q, T) = 0 for all q
V_grid[-1, :] = 0

# Backward Iteration to solve the HJB 
# Loop backward in time from t = T-1, 
# since T is a boundary condition, and up to t=1 only
# We start filling up Vq_{t=T-2} to Vq_{t=0}
for t in range(N_t-1, 0, -1): # 9-1
    # V(q, t) values at the current time step (which is T for i=N_t)
    V_current = V_grid[t].copy() 

    # --- Numerical Approximation of Derivatives ---
    # We need dV/dq at time t (V_t) to solve for V at t-1.
    # Using central difference for dV/dq requires values from adjacent q points.
    # For boundary q points, we'll use forward/backward differences.

    # dV/dq for V_current (dV/dq at time t_i)
    dV_dq_t = np.zeros(N_q)
    # Central difference for interior points
    dV_dq_t[1:-1] = (V_current[2:] - V_current[:-2]) / (2 * dq)
    # Forward difference at the first point (q = -q_max)
    dV_dq_t[0] = (V_current[1] - V_current[0]) / dq
    # Backward difference at the last point (q = +q_max)
    dV_dq_t[-1] = (V_current[-1] - V_current[-2]) / dq
    
    # Recover and calculate optimal trading rate v*(q,t) for the current t.
    # This v* corresponds to the derivative dV/dq
    # v^*(q, t) = (1 / (2*eta)) * (dV/dq)
    V_star[t, :] = (1 / (2 * eta)) * dV_dq_t

    # Solve for V(q, t_{i-1})
    # From the HJB equation: V(q, t_{i-1}) approx V_next(q)
    # And: (V_next(q) - V_current(q)) / dt approx (1 / (4*eta)) * (dV/dq)^2 - gamma*sigma^2*q^2
    # Rearranging to solve for V_next(q):
    # V_next(q) approx V_current(q) + dt * [ (1 / (4*eta)) * (dV/dq_at_current)^2 - gamma*sigma^2*q^2 ]
    
    # Construct the next State Value/Cost Value backward in time V(q, t) -> V(q, t-dt)
    # using the derived PDE dV/dt=[1/(4*eta)](dV/dq)**2 - (gamma*sigma**2q**2)
    """
    # Using the derived form of the HJB equation
    for j in range(N_q):
        ## 1. Get the curremt inventory
        q = inventory_grid[j]
        ## 2. Get the optimal policy, v*(q,t)
        dV_dq = dV_dq_t[j]
        ## 3. Calculate the Hamiltonian
        # Calculate the dV/dt = -H_min(q,t)
        dV_dt = (1 / (4 * eta)) * (dV_dq**2) - (gamma * sigma**2 * q**2)
        ## 4. Update the value function V(q, t) using the HJB equation backwards in time:
        # V(q,t) - V(q, t-dt) = - H_min(q, t) * dt
        # V(q, t-dt) = V(q, t) - H_min(q, t) * dt
        V_grid[t-1, j] = V_current[j] - dV_dt * dt
    """
    # Construct the next State Value/Cost Value backward in time V(q, t) -> V(q, t-dt)
    # using the Hamiltonian(min_v) and HJB equation(dV/dt) equations
    for j in range(N_q):
        ## 1. Get the current inventory
        q = inventory_grid[j]
        ## 2. Get the optimal policy, v*(q,t)
        dV_dq = dV_dq_t[j]
        ## 3. Calculate the Hamiltonian
        #-dVdt = min_{v}{f(v,t) + dVdq*v}
        # where: f(v,t)=eta*v^2 + gamma*sigma^2*q^2
        # Calculate the HJB: 
        #-dVdt = eta*(v_star)**2 + gamma*sigma**2*q**2 - v_star*dVdq
        dV_dt = -(eta * (-V_star[t, j])**2 + gamma*sigma**2*q**2 - V_star[t, j]*dV_dq)
        ## 4. Update the value function V(q, t) using the HJB equation backwards in time:
        # V(q,t) - V(q, t-dt) = - H_min(q, t) * dt
        # V(q, t-dt) = V(q, t) - H_min(q, t) * dt
        V_grid[t-1, j] = V_current[j] - dV_dt * dt

In [119]:
#
# Plot numerical approximation of V(q,t)
#
#print([i for i in V_grid[-2,:]])
#print("time_grid", np.shape(time_grid))
#print("inventory_grid", np.shape(inventory_grid))
X, Y = np.meshgrid(time_grid, inventory_grid) # Q for inventory, T_mesh for time
Z = V_grid.T
#print("X", np.shape(X))
#print("Y", np.shape(Y))
#print("Z", np.shape(Z))
assert np.shape(Z)[0] == np.shape(X)[0]
assert np.shape(Z)[1] == np.shape(Y)[1]

fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(111, projection='3d')
# Plot the surface
ax.plot_surface(X, Y, Z, cmap='viridis', edgecolor='none')
#surface = ax.plot_surface(time_grid, inventory_grid, Z, cmap='viridis', edgecolor='none')
# --- Labels and Title ---
ax.set_xlabel('Time, $\Delta t$', fontsize=fontmed)
ax.set_ylabel('Inventory, $q$', fontsize=fontmed)
ax.set_zlabel('Cost-to-go, $V(q,t)$', fontsize=fontmed)
ax.set_title('State Value, $V(q,t)$\n$Numerical\;Approx$',
             fontsize=fontbig, fontweight="bold", y=.99)
ax.tick_params(axis='both', labelcolor="black", labelsize=fontsmall)
#ax.invert_yaxis()
ax.invert_xaxis()
#ax.invert_zaxis()
#ax.view_init(elev=30, azim=60) # Rotate
ax.set_box_aspect(aspect=None, zoom=0.9)
# Improve layout and display
plt.tight_layout()
plt.savefig(imagepath+"/value_function_numerical.pdf")
if plt_inline_enable:
    plt.show()

In [120]:
### Numerical Approximation of v*(q,t) ####
X, Y = np.meshgrid(time_grid[1:], inventory_grid) # Q for inventory, T_mesh for time
# Calculate V(q, t) for each point in the meshgrid
Z = V_star[1:].T
#print("X", np.shape(X))
#print("Y", np.shape(Y))
#print("Z", np.shape(Z))
assert np.shape(Z)[0] == np.shape(X)[0]
assert np.shape(Z)[1] == np.shape(Y)[1]

#print(V_star[0,:])
fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(111, projection='3d')
surface = ax.plot_surface(X, Y, Z, cmap='viridis', edgecolor='none')
ax.set_xlabel('Time, $\Delta t$', fontsize=fontbig)
ax.set_ylabel('Inventory, $q$', fontsize=fontbig)
ax.set_zlabel('$v^*(q,t)$', fontsize=fontbig)
ax.set_title('Trading Rate, $v^*(q,t)$\n${Numerical\;Approx}$',
             fontsize=fontbig, fontweight="bold", y=.99)
ax.tick_params(axis='both', labelcolor="black", labelsize=fontsmall)
ax.invert_yaxis()
#ax.invert_xaxis()
ax.view_init(elev=30, azim=120) # Rotate
ax.set_box_aspect(aspect=None, zoom=0.9)
# Improve layout and display
plt.tight_layout()
plt.savefig(imagepath+"/control_function_numerical.pdf")
if plt_inline_enable:
    plt.show()